# Biohub Exp100 division-risk prune candidate

Lightweight post-processing candidate that starts from the confirmed Exp073 submission and prunes only the riskiest division second-child edges.

Intentional axis: division precision calibration. Keep all nodes and all ordinary tracking edges; for the top ~5% riskiest division parents by geometry, keep only the closer daughter edge.

This tests whether Exp073 has a small number of false-positive divisions without rerunning detection/tracking.


In [ ]:

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

EXPERIMENT_TAG = "exp100_division_risk_prune_top5pct"
PRUNE_FRAC = 0.05
SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=float)

CANDIDATES = [
    Path("/kaggle/input/biohub-exp073-gap-5-8-public/submission.csv"),
    Path("/kaggle/input/notebooks/dalloliogm/biohub-exp073-gap-5-8-public/submission.csv"),
    Path("kaggle_outputs/biohub-exp073-gap-5-8-public/submission.csv"),
    Path("../kaggle_outputs/biohub-exp073-gap-5-8-public/submission.csv"),
]
INPUT_PATH = next((p for p in CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    discovered = sorted(Path("/kaggle/input").rglob("submission.csv")) if Path("/kaggle/input").exists() else []
    raise FileNotFoundError({"checked": [str(p) for p in CANDIDATES], "discovered": [str(p) for p in discovered[:20]]})

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("kaggle_outputs/biohub-exp100-division-risk-prune-candidate")
WORKING.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = WORKING / "submission.csv"
STATS_PATH = WORKING / "run_stats.csv"
CHECKS_PATH = WORKING / "submission_checks.json"

print("INPUT_PATH", INPUT_PATH)
print("OUTPUT_PATH", OUTPUT_PATH)
print("EXPERIMENT_TAG", EXPERIMENT_TAG)


In [ ]:

df = pd.read_csv(INPUT_PATH)
expected_cols = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
assert list(df.columns) == expected_cols, list(df.columns)

nodes = df[df["row_type"] == "node"].copy()
edges = df[df["row_type"] == "edge"].copy()
node_pos = {
    (str(row.dataset), int(row.node_id)): np.array([float(row.z), float(row.y), float(row.x)], dtype=float) * SCALE_UM
    for row in nodes.itertuples(index=False)
}
node_t = {(str(row.dataset), int(row.node_id)): int(row.t) for row in nodes.itertuples(index=False)}

risk_rows = []
for (dataset, source_id), group in edges.groupby(["dataset", "source_id"], sort=False):
    if len(group) < 2:
        continue
    dataset = str(dataset)
    source_id = int(source_id)
    parent = node_pos[(dataset, source_id)]
    child_records = []
    for row in group.itertuples():
        target_id = int(row.target_id)
        child = node_pos[(dataset, target_id)]
        parent_dist = float(np.linalg.norm(child - parent))
        child_records.append((int(row.Index), target_id, parent_dist, child))
    if len(child_records) != 2:
        # Exp073 should not have >2 outgoing edges, but keep this deterministic.
        child_records = sorted(child_records, key=lambda x: x[2])[:2]
    sister_um = float(np.linalg.norm(child_records[0][3] - child_records[1][3]))
    max_parent_um = max(child_records[0][2], child_records[1][2])
    min_parent_um = min(child_records[0][2], child_records[1][2])
    sum_parent_um = child_records[0][2] + child_records[1][2]
    # Dimensionless risk: favor pruning asymmetric, wide daughter pairs.
    risk = 0.50 * max_parent_um + 0.35 * sister_um + 0.15 * (max_parent_um - min_parent_um)
    keep_edge_index = min(child_records, key=lambda x: x[2])[0]
    drop_edge_index = max(child_records, key=lambda x: x[2])[0]
    risk_rows.append({
        "dataset": dataset,
        "source_id": source_id,
        "keep_edge_index": int(keep_edge_index),
        "drop_edge_index": int(drop_edge_index),
        "risk": risk,
        "max_parent_um": max_parent_um,
        "min_parent_um": min_parent_um,
        "sister_um": sister_um,
        "sum_parent_um": sum_parent_um,
    })

risk_df = pd.DataFrame(risk_rows).sort_values("risk", ascending=False).reset_index(drop=True)
n_divisions = len(risk_df)
n_prune = max(1, int(round(n_divisions * PRUNE_FRAC)))
pruned = risk_df.head(n_prune).copy()
drop_edge_indices = set(pruned["drop_edge_index"].astype(int).tolist())

print("division_like_sources", n_divisions)
print("prune_edges", len(drop_edge_indices))
print(pruned[["dataset", "source_id", "risk", "max_parent_um", "sister_um", "drop_edge_index"]].head(20).to_string(index=False))

calibrated = df.drop(index=sorted(drop_edge_indices)).copy().reset_index(drop=True)
calibrated["id"] = np.arange(len(calibrated), dtype=int)
calibrated.to_csv(OUTPUT_PATH, index=False)

stats = pd.DataFrame([{
    "experiment_tag": EXPERIMENT_TAG,
    "input_path": str(INPUT_PATH),
    "input_rows": int(len(df)),
    "output_rows": int(len(calibrated)),
    "input_nodes": int((df.row_type == "node").sum()),
    "output_nodes": int((calibrated.row_type == "node").sum()),
    "input_edges": int((df.row_type == "edge").sum()),
    "output_edges": int((calibrated.row_type == "edge").sum()),
    "input_division_like_sources": int(n_divisions),
    "pruned_division_edges": int(len(drop_edge_indices)),
    "output_division_like_sources": int(n_divisions - len(drop_edge_indices)),
    "prune_frac": float(PRUNE_FRAC),
    "risk_min_pruned": float(pruned.risk.min()),
    "risk_max_pruned": float(pruned.risk.max()),
}])
stats.to_csv(STATS_PATH, index=False)
print(stats.to_string(index=False))


In [ ]:

out = pd.read_csv(OUTPUT_PATH)
nodes = out[out["row_type"] == "node"].copy()
edges = out[out["row_type"] == "edge"].copy()
node_keys = set(zip(nodes.dataset.astype(str), nodes.node_id.astype(int)))
node_t = {(str(row.dataset), int(row.node_id)): int(row.t) for row in nodes.itertuples(index=False)}
checks = {
    "experiment_tag": EXPERIMENT_TAG,
    "sha256": hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest(),
    "rows": int(len(out)),
    "nodes": int(len(nodes)),
    "edges": int(len(edges)),
    "null_cells": int(out.isna().sum().sum()),
    "id_unique": bool(out["id"].is_unique),
    "id_contiguous": bool((out["id"].to_numpy() == np.arange(len(out))).all()),
    "duplicate_node_keys": int(nodes.duplicated(["dataset", "node_id"]).sum()),
    "missing_edge_sources": int(sum((str(d), int(s)) not in node_keys for d, s in zip(edges.dataset, edges.source_id))),
    "missing_edge_targets": int(sum((str(d), int(t)) not in node_keys for d, t in zip(edges.dataset, edges.target_id))),
    "nonconsecutive_edges": int(sum(node_t.get((str(d), int(tgt)), -999) != node_t.get((str(d), int(src)), -999) + 1 for d, src, tgt in zip(edges.dataset, edges.source_id, edges.target_id))),
    "max_indegree": int(edges.groupby(["dataset", "target_id"]).size().max()) if len(edges) else 0,
    "max_outdegree": int(edges.groupby(["dataset", "source_id"]).size().max()) if len(edges) else 0,
    "division_like_sources": int((edges.groupby(["dataset", "source_id"]).size() >= 2).sum()) if len(edges) else 0,
}
CHECKS_PATH.write_text(json.dumps(checks, indent=2, sort_keys=True) + "\n")
print(json.dumps(checks, indent=2, sort_keys=True))
assert checks["null_cells"] == 0
assert checks["id_unique"] and checks["id_contiguous"]
assert checks["duplicate_node_keys"] == 0
assert checks["missing_edge_sources"] == 0 and checks["missing_edge_targets"] == 0
assert checks["nonconsecutive_edges"] == 0
assert checks["max_indegree"] <= 1
assert checks["max_outdegree"] <= 2
